# 01 - Data Audit and Profiling
## Schema-declared load and a full profile of the raw extract

**Ahsanullah University of Science and Technology** - Department of Computer Science and Engineering

**Course:** CSE 4262 Data Analytics Lab | **Lab Group:** Gr-03 | **Group:** Gr-06

| Student ID | Name |
|---|---|
| 20220104006 | A.S.M. Tahsin Tajware |
| 20220104014 | Abdullah Al Tamim |
| 20220104032 | Eusha Ahmed Mahi |


### Purpose

Load the extract under an explicitly declared schema and profile it **before** any cleaning
touches it. Profiling first matters for a `groupBy`-heavy pipeline, because a null grouping key
silently disappears from an aggregation and the loss is never visible downstream.

Outputs: Tables 1 to 6 of the proposal, and Figure 1.


In [ ]:
import os, sys, glob

_candidates = ["/kaggle/working/repo", "/kaggle/working", "..", "."] + [
    os.path.dirname(p) for p in glob.glob("/kaggle/input/**/da_common.py", recursive=True)]
for _p in _candidates:
    if os.path.exists(os.path.join(_p, "da_common.py")):
        sys.path.insert(0, os.path.abspath(_p))
        break
else:
    raise FileNotFoundError("da_common.py not found. See KAGGLE_SETUP.md for the two setup options.")

from da_common import *

banner("Notebook 01 - Data Audit and Profiling")
spark = get_spark("01 data audit")
print("Spark version :", spark.version)
print("Cores         :", spark.sparkContext.defaultParallelism)

### 1. Load under a declared schema

In [ ]:
raw = read_raw(spark).cache()
n_raw = raw.count()
print(f"Rows loaded: {n_raw:,}")
raw.printSchema()

In [ ]:
raw.select("Category", "rating", "verified_purchase", "helpful_vote",
           F.substring("title", 1, 30).alias("title"),
           F.substring("text", 1, 55).alias("text_preview")).show(5, truncate=False)

### 2. Table 1 - Dataset summary

`None`, `NaN`, `NULL` and empty strings are all counted as missing, matching how pandas reads the
same file, so these counts are directly comparable with the proposal.

In [ ]:
def missing(col):
    c = F.col(col)
    return c.isNull() | c.isin(NA_TOKENS)

s = raw.select(
    F.count("*").alias("total"),
    F.countDistinct("user_id").alias("reviewers"),
    F.countDistinct("asin").alias("products"),
    F.countDistinct("parent_asin").alias("parent_products"),
    F.sum(F.col("verified_purchase").cast("int")).alias("verified"),
    F.sum((~F.col("verified_purchase")).cast("int")).alias("non_verified"),
    F.sum(missing("text").cast("int")).alias("missing_text"),
    F.sum(missing("title").cast("int")).alias("missing_title"),
    F.min((F.col("timestamp") / 1000).cast("timestamp")).alias("earliest"),
    F.max((F.col("timestamp") / 1000).cast("timestamp")).alias("latest"),
).first()

tot = s["total"]
tbl1 = pd.DataFrame([
    ("Total records",           f"{tot:,} customer reviews"),
    ("Date range",              f"{s['earliest'].date()} to {s['latest'].date()}"),
    ("Categories",              ", ".join(CATS)),
    ("Verified / non-verified", f"{s['verified']:,} ({s['verified']/tot:.1%}) / "
                                f"{s['non_verified']:,} ({s['non_verified']/tot:.1%})"),
    ("Unique reviewers",        f"{s['reviewers']:,}"),
    ("Unique products (asin)",  f"{s['products']:,}"),
    ("Unique parent products",  f"{s['parent_products']:,}"),
    ("Missing review text",     f"{s['missing_text']} ({s['missing_text']/tot:.3%})"),
    ("Missing review title",    f"{s['missing_title']} ({s['missing_title']/tot:.3%})"),
], columns=["Attribute", "Value"])

save_table(tbl1, "tbl01_dataset_summary")
tbl1

### 3. Tables 2 and 3 - Category and rating distribution

In [ ]:
cat_dist = (raw.groupBy("Category").count()
            .withColumn("share_pct", F.round(100 * F.col("count") / tot, 1))
            .orderBy(F.desc("count")))
rating_dist = (raw.groupBy("rating").count()
               .withColumn("share_pct", F.round(100 * F.col("count") / tot, 1))
               .orderBy("rating"))

print("Table 2 - Category distribution"); cat_dist.show(truncate=False)
print("Table 3 - Rating distribution");   rating_dist.show(truncate=False)

save_table(cat_dist, "tbl02_category_distribution")
save_table(rating_dist, "tbl03_rating_distribution");

### 4. Tables 4 and 5 - Rating by category, length by rating

In [ ]:
cat_stats = (raw.groupBy("Category")
             .agg(F.round(F.avg("rating"), 2).alias("avg_rating"),
                  F.round(100 * F.avg(F.col("verified_purchase").cast("int")), 1).alias("verified_share_pct"),
                  F.count("*").alias("reviews"))
             .orderBy(F.desc("avg_rating")))

len_by_rating = (raw.withColumn("len", F.length(F.coalesce(F.col("text"), F.lit(""))))
                 .groupBy("rating")
                 .agg(F.round(F.avg("len")).cast("int").alias("avg_len_chars"))
                 .orderBy("rating"))

print("Table 4 - Average rating and verified share by category"); cat_stats.show(truncate=False)
print("Table 5 - Average review length by star rating");          len_by_rating.show(truncate=False)

save_table(cat_stats, "tbl04_category_stats")
save_table(len_by_rating, "tbl05_length_by_rating");

### 5. Table 6 - Helpful-vote engagement profile

Helpful votes are the most skewed field in the extract. The median and the 75th percentile both
sit at zero while a few reviews collect thousands, so the mean on its own is misleading and any
helpfulness feature has to be built around that skew.

In [ ]:
q = raw.approxQuantile("helpful_vote", [0.5, 0.75, 0.9, 0.99], 0.001)
hv = raw.select(F.round(F.avg("helpful_vote"), 3).alias("mean"),
                F.max("helpful_vote").alias("max"),
                F.round(100 * F.avg((F.col("helpful_vote") >= 1).cast("int")), 1).alias("pct_any")).first()

tbl6 = pd.DataFrame([
    ("Mean helpful votes", hv["mean"]), ("Median (p50)", q[0]), ("75th percentile", q[1]),
    ("90th percentile", q[2]), ("99th percentile", q[3]), ("Maximum", hv["max"]),
    ("Reviews with at least one vote (%)", hv["pct_any"]),
], columns=["Statistic", "Value"])

save_table(tbl6, "tbl06_helpful_vote_profile")
tbl6

### 6. Figure 1 - Profiling overview

In [ ]:
rd, cd, lbr = rating_dist.toPandas(), cat_dist.toPandas(), len_by_rating.toPandas()
fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))

bars = ax[0].bar(rd["rating"].astype(int).astype(str), rd["count"], color=STAR_COLORS)
ax[0].set_title("(a) Rating distribution is J-shaped")
ax[0].set_xlabel("Star rating"); ax[0].set_ylabel("Number of reviews")
ax[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
for b, s_ in zip(bars, rd["share_pct"]):
    ax[0].text(b.get_x() + b.get_width()/2, b.get_height(), f"{s_}%", ha="center", va="bottom", fontsize=9)

ax[1].bar(cd["Category"], cd["count"], color=[ccol(c) for c in cd["Category"]])
ax[1].set_title("(b) Reviews per category"); ax[1].set_ylabel("Number of reviews")
ax[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
for i, (c, s_) in enumerate(zip(cd["count"], cd["share_pct"])):
    ax[1].text(i, c, f"{s_}%", ha="center", va="bottom", fontsize=9)

ax[2].plot(lbr["rating"].astype(int), lbr["avg_len_chars"], marker="o", color="#db2777", lw=2)
ax[2].set_title("(c) Average review length vs rating")
ax[2].set_xlabel("Star rating"); ax[2].set_ylabel("Average length (characters)")
ax[2].set_xticks([1, 2, 3, 4, 5])
for x, y in zip(lbr["rating"].astype(int), lbr["avg_len_chars"]):
    ax[2].text(x, y, f"{y}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
savefig(fig, "fig01_dataset_profile", "Rating skew, category mix, length by rating")
plt.show()

### Findings

Three properties of the extract shape every decision downstream. Ratings are J-shaped rather than
bell-shaped, so an average rating hides the distribution and product ranking will need a minimum
review threshold. Categories are unevenly represented, so per-category results carry unequal
weight and review counts must be reported next to every ranking. And helpful votes reach only a
small minority of reviews, so helpfulness behaves more like a rare event than a continuous score.


In [ ]:
print(f"Tables in {TBL_DIR}")
for name in sorted(RESULTS):
    print(f"  {name:<40} {len(RESULTS[name]):>6,} rows")
print(f"\nFigures in {FIG_DIR}")
for name in sorted(FIGURES):
    print(f"  {name}.png")

In [ ]:
spark.stop()
print("Spark session stopped.")